In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, Field, computed_field, ConfigDict

class Ball(BaseModel):
    """a 3d ball with material and state"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    radius: float
    density: float
    color: str = "blue"
    elasticity: float
    max_deformation: float = 0.1
    stiffness: float = 500.0
    dilation: float = 0.0

    start_position: list[float]
    start_velocity: list[float]
    current_position: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    current_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    angular_velocity: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    acceleration: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    force: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))
    torque: np.ndarray = Field(default_factory=lambda: np.zeros(3, dtype=float))

    @computed_field
    @property
    def mass(self) -> float:
        return (4.0 / 3.0) * np.pi * (self.radius ** 3) * self.density

    @computed_field
    @property
    def moment_of_inertia(self) -> float:
        return (2.0 / 5.0) * self.mass * (self.radius ** 2)

    def initialize_state(self) -> None:
        self.current_position = np.array(self.start_position, dtype=float)
        self.current_velocity = np.array(self.start_velocity, dtype=float)
        self.angular_velocity = np.zeros(3, dtype=float)
        self.acceleration = np.zeros(3, dtype=float)
        self.force = np.zeros(3, dtype=float)
        self.torque = np.zeros(3, dtype=float)
        self.dilation = 0.0


medium_types = Literal["air", "helium", "nitrogen"]
class BounceEnvironment(BaseModel):
    """environment parameters for the room"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    room_dimensions: list[float]
    gravity: float
    gravity_vector: list[float] = [0.0, 0.0, -1.0]
    linear_drag: float = 0.0
    fluid_density: float = 1.225
    drag_coefficient: float = 0.47
    medium_type: medium_types = "air"
    wall_restitution: float = 0.9
    wall_curvature: float = 0.0


In [2]:

class Simulation(BaseModel):
    """run a physics simulation for multiple balls"""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    balls: list[Ball]
    environment: BounceEnvironment
    time_step: float
    total_time_steps: int
    positions: list[np.ndarray] = Field(default_factory=list)
    velocities: list[np.ndarray] = Field(default_factory=list)
    occupancy_grid: np.ndarray | None = None
    occupancy_history: list[np.ndarray] = Field(default_factory=list)

    def _gravity_vector(self) -> np.ndarray:
        g_dir = np.array(self.environment.gravity_vector, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            return np.zeros(3, dtype=float)
        return (g_dir / g_norm) * self.environment.gravity

    def calculate_forces(self) -> None:
        g_vec = self._gravity_vector()
        env = self.environment
        for ball in self.balls:
            ball.force = ball.mass * g_vec
            ball.torque = np.zeros(3, dtype=float)
            v = ball.current_velocity
            speed = np.linalg.norm(v)
            omega = ball.angular_velocity
            omega_mag = np.linalg.norm(omega)

            # linear drag
            if env.linear_drag != 0.0:
                ball.force += -env.linear_drag * v

            # quadratic drag
            if speed > 0.0 and env.fluid_density > 0.0:
                area = np.pi * (ball.radius ** 2)
                quad_mag = 0.5 * env.fluid_density * env.drag_coefficient * area * speed
                ball.force += -quad_mag * v

            # magnus force: lift from spin in fluid
            if speed > 0.0 and omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_magnus = 0.5
                ball.force += c_magnus * env.fluid_density * (ball.radius ** 3) * np.cross(omega, v)

            # angular drag torque: viscous resistance to spin
            if omega_mag > 1e-10 and env.fluid_density > 0.0:
                c_ang_drag = 0.1
                ball.torque += -c_ang_drag * env.fluid_density * (ball.radius ** 5) * omega_mag * omega

    def update_acceleration(self) -> None:
        for ball in self.balls:
            ball.acceleration = ball.force / ball.mass

    def update_velocity(self) -> None:
        max_speed = 50.0
        max_spin = 100.0
        for ball in self.balls:
            v_new = ball.current_velocity + ball.acceleration * self.time_step
            omega_new = ball.angular_velocity + (ball.torque / ball.moment_of_inertia) * self.time_step

            if not np.all(np.isfinite(v_new)):
                v_new = np.zeros(3, dtype=float)
            if not np.all(np.isfinite(omega_new)):
                omega_new = np.zeros(3, dtype=float)

            speed = np.linalg.norm(v_new)
            if speed > max_speed:
                v_new = v_new * (max_speed / speed)

            omega_mag = np.linalg.norm(omega_new)
            if omega_mag > max_spin:
                omega_new = omega_new * (max_spin / omega_mag)

            ball.current_velocity = v_new
            ball.angular_velocity = omega_new

    def update_position(self) -> None:
        for ball in self.balls:
            ball.current_position = (
                ball.current_position + ball.current_velocity * self.time_step
            )

    def update_deformation(self) -> None:
        recovery_rate = 2.0
        for ball in self.balls:
            ball.dilation = max(0.0, ball.dilation - recovery_rate * self.time_step)

    def bounce_off_walls(self, ball: Ball) -> bool:
        dims = np.array(self.environment.room_dimensions, dtype=float)
        pos = ball.current_position.copy()
        vel = ball.current_velocity.copy()
        omega = ball.angular_velocity.copy()
        r = ball.radius
        e_eff = ball.elasticity * self.environment.wall_restitution
        hit = False
        wall_friction = 0.2
        friction_applied = False

        for axis in range(3):
            min_bound = r
            max_bound = dims[axis] - r
            e_axis = np.zeros(3, dtype=float)
            e_axis[axis] = 1.0

            colliding_low = pos[axis] < min_bound and vel[axis] < 0
            colliding_high = pos[axis] > max_bound and vel[axis] > 0

            if colliding_low or colliding_high:
                # save incoming velocity for dilation and friction before modifying
                incoming_speed = abs(vel[axis])
                r_vec = -r * e_axis if colliding_low else r * e_axis

                # compute friction using incoming velocity
                if not friction_applied and ball.mass > 0 and ball.moment_of_inertia > 0:
                    v_cp = vel + np.cross(omega, r_vec)
                    v_tang = v_cp - np.dot(v_cp, e_axis) * e_axis
                    v_tang_mag = np.linalg.norm(v_tang)
                    if v_tang_mag > 1e-10:
                        j_t = -wall_friction * v_tang
                        vel += j_t / ball.mass
                        omega += np.cross(r_vec, j_t) / ball.moment_of_inertia
                        friction_applied = True

                # apply normal impulse (bounce)
                pos[axis] = min_bound if colliding_low else max_bound
                vel[axis] = -vel[axis] * e_eff
                ball.dilation = min(ball.max_deformation, ball.dilation + incoming_speed * 0.01)
                hit = True

        ball.current_position = pos
        ball.current_velocity = vel
        ball.angular_velocity = omega
        return hit

    def handle_ball_collisions(self) -> bool:
        count = len(self.balls)
        had_collision = False

        for i in range(count):
            for j in range(i + 1, count):
                b1 = self.balls[i]
                b2 = self.balls[j]
                delta = b2.current_position - b1.current_position
                dist = np.linalg.norm(delta)
                min_dist = b1.radius + b2.radius

                if dist >= min_dist:
                    continue

                had_collision = True

                if dist == 0.0:
                    normal = np.array([1.0, 0.0, 0.0], dtype=float)
                else:
                    normal = delta / dist

                overlap = min_dist - dist
                inv_m1 = 1.0 / b1.mass if b1.mass > 0.0 else 0.0
                inv_m2 = 1.0 / b2.mass if b2.mass > 0.0 else 0.0
                total_inv_mass = inv_m1 + inv_m2
                if total_inv_mass > 0.0:
                    b1.current_position -= normal * (overlap * (inv_m1 / total_inv_mass))
                    b2.current_position += normal * (overlap * (inv_m2 / total_inv_mass))

                rel_vel = b2.current_velocity - b1.current_velocity
                vel_along_normal = np.dot(rel_vel, normal)
                if vel_along_normal > 0.0:
                    continue

                # dilation based on impact velocity, not overlap
                impact_speed = abs(vel_along_normal)
                b1.dilation = min(b1.max_deformation, b1.dilation + impact_speed * 0.01)
                b2.dilation = min(b2.max_deformation, b2.dilation + impact_speed * 0.01)

                e_eff = np.sqrt(b1.elasticity * b2.elasticity)
                impulse_mag = (-(1.0 + e_eff) * vel_along_normal) / total_inv_mass
                impulse = impulse_mag * normal

                b1.current_velocity -= impulse * inv_m1
                b2.current_velocity += impulse * inv_m2

                # tangential friction impulse (produces torque on both balls)
                ball_friction = 0.3
                r1_cp = b1.radius * normal
                r2_cp = -b2.radius * normal
                v_cp = ((b1.current_velocity + np.cross(b1.angular_velocity, r1_cp))
                        - (b2.current_velocity + np.cross(b2.angular_velocity, r2_cp)))
                v_t = v_cp - np.dot(v_cp, normal) * normal
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag > 1e-10:
                    t_hat = v_t / v_t_mag
                    r1xt = np.cross(r1_cp, t_hat)
                    r2xt = np.cross(r2_cp, t_hat)
                    inv_m_eff_t = (inv_m1 + inv_m2
                                   + np.dot(r1xt, r1xt) / b1.moment_of_inertia
                                   + np.dot(r2xt, r2xt) / b2.moment_of_inertia)
                    j_t_needed = v_t_mag / inv_m_eff_t if inv_m_eff_t > 0 else 0.0
                    j_t = min(j_t_needed, ball_friction * abs(impulse_mag))
                    fric = -j_t * t_hat

                    b1.current_velocity += fric * inv_m1
                    b2.current_velocity -= fric * inv_m2
                    b1.angular_velocity += np.cross(r1_cp, fric) / b1.moment_of_inertia
                    b2.angular_velocity -= np.cross(r2_cp, fric) / b2.moment_of_inertia

        return had_collision

    def apply_rolling_friction(self) -> None:
        """friction and torque for balls resting on walls"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        wall_friction = 0.3
        contact_tol = 0.05

        for ball in self.balls:
            r = ball.radius
            pos = ball.current_position
            vel = ball.current_velocity
            omega = ball.angular_velocity

            for axis in range(3):
                in_contact = False
                n = np.zeros(3, dtype=float)

                if pos[axis] <= r + contact_tol:
                    n[axis] = 1.0
                    in_contact = True
                elif pos[axis] >= dims[axis] - r - contact_tol:
                    n[axis] = -1.0
                    in_contact = True

                if not in_contact:
                    continue

                r_vec = -r * n
                v_cp = vel + np.cross(omega, r_vec)
                v_t = v_cp - np.dot(v_cp, n) * n
                v_t_mag = np.linalg.norm(v_t)

                if v_t_mag < 1e-10:
                    continue

                f_normal = max(0.0, -np.dot(ball.force, n))
                if f_normal < 1e-10:
                    continue

                f_fric = wall_friction * f_normal
                impulse_mag = f_fric * self.time_step
                impulse_mag = min(impulse_mag, ball.mass * v_t_mag * 0.5)

                t_hat = v_t / v_t_mag
                fric_impulse = -impulse_mag * t_hat

                ball.current_velocity += fric_impulse / ball.mass
                ball.angular_velocity += np.cross(r_vec, fric_impulse) / ball.moment_of_inertia

    def _snapshot(self) -> tuple[np.ndarray, np.ndarray]:
        positions = np.stack([b.current_position for b in self.balls], axis=0)
        velocities = np.stack([b.current_velocity for b in self.balls], axis=0)
        return positions, velocities

    def init_grid(self, resolution: int = 20) -> None:
        self.occupancy_grid = np.zeros((resolution, resolution, resolution), dtype=float)

    def update_occupancy(self) -> None:
        if self.occupancy_grid is None:
            return
        dims = np.array(self.environment.room_dimensions, dtype=float)
        res = self.occupancy_grid.shape[0]
        for ball in self.balls:
            if not np.all(np.isfinite(ball.current_position)):
                continue
            idx = ((ball.current_position / dims) * (res - 1)).astype(int)
            idx = np.clip(idx, 0, res - 1)
            self.occupancy_grid[tuple(idx)] += 1.0

    def clamp_to_bounds(self) -> None:
        """position-only clamp after ball-ball separation"""
        dims = np.array(self.environment.room_dimensions, dtype=float)
        for ball in self.balls:
            r = ball.radius
            ball.current_position = np.clip(
                ball.current_position, r, dims - r,
            )

    def step(self, store: bool = True) -> None:
        self.calculate_forces()
        self.update_acceleration()
        self.update_velocity()
        self.update_position()
        self.update_deformation()
        for ball in self.balls:
            self.bounce_off_walls(ball)
        self.handle_ball_collisions()
        self.apply_rolling_friction()
        self.clamp_to_bounds()
        if store:
            positions, velocities = self._snapshot()
            self.positions.append(positions)
            self.velocities.append(velocities)
        self.update_occupancy()
        if store and self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())

    def simulate(self) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for _ in range(self.total_time_steps):
            self.step()

    def simulate_until(self, stop_event, max_steps: int = 500_000, store_every: int = 10, max_frames: int = 3000) -> None:
        for ball in self.balls:
            ball.initialize_state()
        positions, velocities = self._snapshot()
        self.positions = [positions]
        self.velocities = [velocities]
        self.init_grid()
        self.occupancy_history = []
        self.update_occupancy()
        if self.occupancy_grid is not None:
            self.occupancy_history.append(self.occupancy_grid.sum(axis=2).copy())
        for i in range(max_steps):
            if stop_event.is_set():
                break
            store = (i % store_every == 0) and len(self.positions) < max_frames
            self.step(store=store)

In [4]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import ast


def _random_non_overlapping_positions(radii, room_dims, max_tries=5000):
    rng = np.random.default_rng()
    positions = []

    for r in radii:
        mins = np.array([r, r, r], dtype=float)
        maxs = np.array(room_dims, dtype=float) - r
        placed = False

        for _ in range(max_tries):
            candidate = rng.uniform(mins, maxs)
            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} without overlap")
    return positions


def _initial_positions(radii, room_dims, layout="random", max_tries=5000):
    """generate initial positions for different layouts while avoiding overlap"""
    rng = np.random.default_rng()
    dims = np.array(room_dims, dtype=float)

    if layout == "random":
        return _random_non_overlapping_positions(radii, room_dims, max_tries=max_tries)

    positions = []
    center = dims / 2.0

    for idx, r in enumerate(radii):
        placed = False
        for _ in range(max_tries):
            if layout == "cluster_center":
                spread = dims.min() * 0.2
                candidate = center + rng.normal(scale=spread, size=3)
            elif layout == "cluster_corner":
                base = np.array([r, r, r], dtype=float)
                spread = dims.min() * 0.2
                candidate = base + rng.normal(scale=spread, size=3)
            elif layout == "line":
                t = idx / max(1, len(radii) - 1)
                candidate = np.array([
                    r + t * (dims[0] - 2 * r),
                    dims[1] * 0.5,
                    dims[2] * 0.5,
                ], dtype=float)
            elif layout == "high_drop":
                candidate = np.array([
                    rng.uniform(r, dims[0] - r),
                    rng.uniform(r, dims[1] - r),
                    dims[2] * 0.8,
                ], dtype=float)
            else:
                candidate = rng.uniform([r, r, r], dims - r)

            if np.any(candidate < r) or np.any(candidate > dims - r):
                continue

            if all(np.linalg.norm(candidate - p) >= (r + p_r)
                   for p, p_r in zip(positions, radii[:len(positions)])):
                positions.append(candidate)
                placed = True
                break
        if not placed:
            raise ValueError(f"could not place ball with radius {r} for layout {layout}")

    return positions


def _build_balls(count, radius_range, room_dims, speed, density, layout="random", elasticity=0.5, color="blue"):
    rng = np.random.default_rng()
    r_min, r_max = radius_range
    radii = rng.uniform(r_min, r_max, count)

    positions = _initial_positions(radii, room_dims, layout=layout)

    balls = []
    for i in range(count):
        r = radii[i]
        m = density * (4.0 / 3.0) * np.pi * (r ** 3)

        direction = rng.normal(size=3)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0.0:
            direction = np.array([1.0, 0.0, 0.0], dtype=float)
            direction_norm = 1.0
        velocity = (direction / direction_norm) * speed

        balls.append(
            Ball(
                radius=r,
                mass=m,
                color=color,
                start_position=positions[i].tolist(),
                start_velocity=velocity.tolist(),
                elasticity=elasticity,
                density=density,
            )
        )
    return balls


def _compute_scalar_fields(positions, velocities, balls, env, mode="velocity"):
    values_per_frame = []

    if mode == "velocity":
        for v in velocities:
            values_per_frame.append(np.linalg.norm(v, axis=1))
    elif mode == "potential_energy":
        g_dir = np.array(env.gravity_direction, dtype=float)
        g_norm = np.linalg.norm(g_dir)
        if g_norm == 0.0:
            g_dir = np.array([0.0, 0.0, -1.0], dtype=float)
            g_norm = 1.0
        unit_g = g_dir / g_norm
        masses = np.array([b.mass for b in balls], dtype=float)

        for pos in positions:
            height = -np.dot(pos, unit_g)
            pe = masses * env.gravity * height
            values_per_frame.append(pe)
    else:
        for pos in positions:
            values_per_frame.append(np.zeros(pos.shape[0], dtype=float))

    all_vals = np.concatenate(values_per_frame) if values_per_frame else np.array([0.0])
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if vmax <= vmin:
        vmax = vmin + 1.0

    return values_per_frame, vmin, vmax


def _plot_with_density(sim, balls, env, room_dims, frame_stride=1, color_mode="velocity"):
    positions = sim.positions
    velocities = sim.velocities
    dims = np.array(room_dims, dtype=float)

    scene_config = dict(
        xaxis=dict(range=[0, dims[0]], autorange=False),
        yaxis=dict(range=[0, dims[1]], autorange=False),
        zaxis=dict(range=[0, dims[2]], autorange=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=dims[1] / dims[0], z=dims[2] / dims[0]),
    )

    radii = np.array([b.radius for b in balls], dtype=float)
    if radii.size == 0:
        marker_sizes = 6
    else:
        r_max = radii.max()
        if r_max <= 0.0:
            marker_sizes = 6
        else:
            marker_sizes = 4.0 + (radii / r_max) * 8.0

    scalar_values, vmin, vmax = _compute_scalar_fields(positions, velocities, balls, env, mode=color_mode)

    initial = positions[0]
    initial_vals = scalar_values[0]

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=initial[:, 0],
                y=initial[:, 1],
                z=initial[:, 2],
                mode="markers",
                marker=dict(
                    size=marker_sizes,
                    color=initial_vals,
                    colorscale="Viridis",
                    cmin=vmin,
                    cmax=vmax,
                    opacity=0.8,
                    colorbar=dict(title=color_mode),
                ),
            )
        ]
    )

    frames = []
    for t in range(0, len(positions), frame_stride):
        pos = positions[t]
        vals = scalar_values[t]
        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=pos[:, 0],
                        y=pos[:, 1],
                        z=pos[:, 2],
                        mode="markers",
                        marker=dict(
                            size=marker_sizes,
                            color=vals,
                            colorscale="Viridis",
                            cmin=vmin,
                            cmax=vmax,
                            opacity=0.8,
                        ),
                    )
                ],
                name=str(t),
            )
        )

    fig.update(frames=frames)

    fig.update_layout(
        scene=scene_config,
        uirevision="constant_view",
        margin=dict(l=0, r=0, b=0, t=30),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="play",
                        method="animate",
                        args=[
                            None,
                            {"frame": {"duration": 30, "redraw": True}, "fromcurrent": True},
                        ],
                    ),
                    dict(
                        label="pause",
                        method="animate",
                        args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}],
                    ),
                ],
            )
        ],
    )

    return fig


ball_count = widgets.IntSlider(value=50, min=1, max=1000, step=1, description="balls")
radius_range = widgets.FloatRangeSlider(
    value=[0.3, 0.8], min=0.1, max=2.0, step=0.1, description="radius range"
)
ball_density = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="density")
ball_speed = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="speed")

room_dims = widgets.Text(
    value="[10.0, 10.0, 10.0]",
    description="room [x,y,z]",
)

gravity = widgets.FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description="gravity")
restitution = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="bounciness")

material = widgets.Dropdown(
    options=["rubber", "steel", "foam"],
    value="rubber",
    description="material",
)

color_mode = widgets.Dropdown(
    options=[
        ("velocity", "velocity"),
        ("potential energy", "potential_energy"),
    ],
    value="velocity",
    description="color by",
)

init_layout = widgets.Dropdown(
    options=[
        ("random", "random"),
        ("cluster center", "cluster_center"),
        ("cluster corner", "cluster_corner"),
        ("line", "line"),
        ("high drop", "high_drop"),
    ],
    value="random",
    description="layout",
)

run_button = widgets.Button(description="generate sim")
output = widgets.Output()


def _run_simulation(_):
    output.clear_output(wait=True)
    with output:
        try:
            dims_val = ast.literal_eval(room_dims.value)
            dims = [float(v) for v in dims_val]
            if len(dims) != 3:
                raise ValueError
        except Exception:
            dims = [10.0, 10.0, 10.0]

        material_props = {
            "rubber": {"elasticity": 0.9, "color": "red"},
            "steel": {"elasticity": 0.6, "color": "gray"},
            "foam": {"elasticity": 0.8, "color": "orange"},
        }
        props = material_props[material.value]
        elasticity = props["elasticity"]
        color = props["color"]

        env = BounceEnvironment(
            room_dimensions=dims,
            gravity=gravity.value,
            gravity_direction=[0.0, 0.0, -1.0],
            linear_drag=0.0,
            quadratic_drag=0.0,
            restitution=restitution.value,
        )
        balls = _build_balls(
            ball_count.value,
            radius_range.value,
            dims,
            ball_speed.value,
            ball_density.value,
            layout=init_layout.value,
            elasticity=elasticity,
            color=color,
        )
        sim = Simulation(
            balls=balls,
            environment=env,
            time_step=0.02,
            total_time_steps=2000,
        )
        sim.simulate()
        frame_stride = max(1, len(sim.positions) // 300)
        fig = _plot_with_density(
            sim,
            balls,
            env,
            dims,
            frame_stride=frame_stride,
            color_mode=color_mode.value,
        )
        display(fig)


run_button.on_click(_run_simulation)

controls = widgets.VBox(
    [
        ball_count,
        radius_range,
        ball_density,
        ball_speed,
        room_dims,
        gravity,
        restitution,
        material,
        color_mode,
        init_layout,
        run_button,
    ]
)

ui = widgets.HBox([controls, output])
display(ui)

In [ ]:
import json
from IPython.display import HTML

def render_threejs(sim):
    # Convert simulation positions to a flat list for JS efficiency
    # Shape: (frames, num_balls, 3) -> List of positions
    pos_data = [p.tolist() for p in sim.positions]
    ball_info = [{"color": b.color, "radius": b.radius} for b in sim.balls]
    dims = sim.environment.room_dimensions

    html_template = f"""
    <div id="threejs-container" style="width: 100%; height: 500px; background: #111;"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script>
    (function() {{
        const container = document.getElementById('threejs-container');
        const width = container.clientWidth;
        const height = 500;
        
        const scene = new THREE.Scene();
        const camera = new THREE.PerspectiveCamera(75, width / height, 0.1, 1000);
        const renderer = new THREE.WebGLRenderer({{ antialias: true }});
        renderer.setSize(width, height);
        container.appendChild(renderer.domElement);

        // Room Box
        const roomGeo = new THREE.BoxGeometry({dims[0]}, {dims[1]}, {dims[2]});
        const roomMat = new THREE.MeshBasicMaterial({{ color: 0x444444, wireframe: true }});
        const room = new THREE.Mesh(roomGeo, roomMat);
        room.position.set({dims[0]/2}, {dims[1]/2}, {dims[2]/2});
        scene.add(room);

        // Lights
        const light = new THREE.PointLight(0xffffff, 1, 100);
        light.position.set(10, 20, 10);
        scene.add(light);
        scene.add(new THREE.AmbientLight(0x404040));

        // Balls
        const ballsData = {json.dumps(ball_info)};
        const frames = {json.dumps(pos_data)};
        const ballMeshes = ballsData.map(info => {{
            const geo = new THREE.SphereGeometry(info.radius, 32, 32);
            const mat = new THREE.MeshPhongMaterial({{ color: info.color }});
            const mesh = new THREE.Mesh(geo, mat);
            scene.add(mesh);
            return mesh;
        }});

        camera.position.set({dims[0]*1.5}, {dims[1]*1.5}, {dims[2]*2});
        camera.lookAt({dims[0]/2}, {dims[1]/2}, {dims[2]/2});

        let frameIdx = 0;
        function animate() {{
            requestAnimationFrame(animate);
            if (frameIdx < frames.length) {{
                const currentFrame = frames[frameIdx];
                ballMeshes.forEach((mesh, i) => {{
                    mesh.position.set(currentFrame[i][0], currentFrame[i][1], currentFrame[i][2]);
                }});
                frameIdx = (frameIdx + 1) % frames.length; // Loop animation
            }}
            renderer.render(scene, camera);
        }}
        animate();
    }})();
    </script>
    """
    return HTML(html_template)

# Run the simulation first, then call:
render_threejs(sim)

NameError: name 'sim' is not defined